# U.S. Hazardous Liquid Pipeline — Exploration Notebook
### Data: PHMSA Annual Reports 2004–2025 + Incident Database 1986–2025

**Story beats being built here:**
1. *"What even is a hazardous liquid pipeline?"* — scale callout card
2. *"Wait — there are 228,000 miles of this stuff?"* — annotated growth line chart (Visual 1A)
3. *"How old is it?"* — decade timeline strip + age tier donut (Visual 3A / 3C)

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import xlrd
import warnings
warnings.filterwarnings('ignore')

BASE = r'c:\Project Clean\PHSMA'
print('Libraries loaded OK')

Libraries loaded OK


---
## 1. Load Mileage Summary (2004–2025)

In [2]:
wb = xlrd.open_workbook(rf'{BASE}\Mileage Summary\Extracted\haz_liquid_annual_stat.xls')
ws = wb.sheet_by_index(0)

rows = []
for r in range(ws.nrows):
    vals = [ws.cell_value(r, c) for c in range(ws.ncols)]
    if vals[1] and str(vals[1]).replace('.0','').isdigit():
        rows.append({
            'year':   int(vals[1]),
            'n_ops':  vals[2],
            'total':  vals[3],
            'rpp':    vals[4],   # Petroleum / Refined Products
            'hvl':    vals[5],   # Highly Volatile Liquids
            'crude':  vals[6],   # Crude Oil
            'co2':    vals[7],   # CO2 / Other
            'ethanol':vals[8] if vals[8] else 0,
        })

miles = pd.DataFrame(rows)
print(f"Years: {miles.year.min()}–{miles.year.max()}  |  Rows: {len(miles)}")
miles.tail(5)

Years: 2004–2025  |  Rows: 22


,year,n_ops,total,rpp,hvl,crude,co2,ethanol
17,2021,696.0,230035.833,64253.409,75601.042,84825.520,5339.240,16.622
18,2022,683.0,229366.487,64136.279,75486.051,84371.580,5354.278,18.299
19,2023,689.0,228758.068,64105.006,75151.716,84148.529,5331.218,21.599
20,2024,664.0,228513.822,64230.518,75708.318,83208.718,5344.649,21.619
21,2025,656.0,227457.052,65067.112,75218.829,81828.510,5320.982,21.619


---
## 2. Visual 1A — Annotated Growth Line Chart

### Famous pipeline events used as annotations

Each annotation has a short blurb that will appear on hover in the final interactive version.
These are the facts behind each callout:

In [3]:
# ── Pipeline event annotations ────────────────────────────────────────────────
# Each entry: (year, label, side, blurb)
# 'side' = 'top' or 'bottom' — controls whether the flag goes above or below the line
# 'blurb' = the companion text shown in tooltip / sidebar story panel

events = [
    (
        2008,
        "Shale boom begins",
        "top",
        "Hydraulic fracturing (fracking) unlocks vast Bakken and Permian Basin "
        "reserves. Crude oil production surges — and pipeline construction races "
        "to catch up. The US crude network will nearly double over the next decade."
    ),
    (
        2010,
        "Kalamazoo River spill — $1.2B cleanup",
        "bottom",
        "Enbridge Line 6B ruptures near Marshall, Michigan, releasing ~20,000 barrels "
        "(840,000 gallons) of diluted bitumen into the Kalamazoo River — the largest "
        "inland oil spill in US Midwest history. The pipeline was 41 years old. "
        "Cleanup cost exceeded $1.2 billion. The river was closed for two years."
    ),
    (
        2015,
        "Refugio spill — $92M cleanup",
        "bottom",
        "A 50-year-old Plains All American pipeline ruptures near Santa Barbara, CA. "
        "Corrosion had worn the pipe wall to less than an inch thick. Over 100,000 gallons "
        "reach the Pacific Ocean, fouling 9 miles of coastline. The company initially "
        "failed to report the spill. Cleanup cost: $92M+. Plains pays $60M in penalties."
    ),
    (
        2017,
        "Dakota Access opens — $3.8B",
        "top",
        "The 1,172-mile Dakota Access Pipeline (DAPL) becomes commercially operational "
        "on June 1, 2017, after the Standing Rock Sioux Tribe's historic protests "
        "drew global attention. The pipeline crosses under Lake Oahe — the tribe's "
        "primary water source. Despite ongoing legal challenges over insufficient "
        "environmental review, it carries 750,000 barrels/day of Bakken crude oil "
        "and remains operational today."
    ),
    (
        2019,
        "Crude network peaks",
        "top",
        "Crude oil pipeline mileage hits its all-time high at 84,475 miles — a 71% "
        "increase from 2004 — driven by the Permian Basin boom. New pipelines include "
        "the Cactus II (Plains All American, 670k bbl/day from Permian to Corpus Christi) "
        "and the EPIC Crude Pipeline. The Permian Basin alone accounts for nearly "
        "half of all US crude production by this point."
    ),
    (
        2021,
        "Colonial ransomware — $4.4M ransom",
        "bottom",
        "A cyberattack shuts down the Colonial Pipeline — the largest fuel pipeline "
        "in the US — for 6 days in May 2021. The 5,500-mile pipeline carries 45% of "
        "the East Coast's fuel supply: gasoline, diesel, and jet fuel. Panic buying "
        "causes shortages at gas stations from Georgia to New Jersey. DarkSide extorted "
        "$4.4M; the FBI recovered $2.3M. The attack exposed how vulnerable invisible "
        "infrastructure is to threats beyond corrosion."
    ),
    (
        2021,
        "Network peaks: 230,036 mi",
        "top",
        "The US hazardous liquid pipeline network reaches its all-time high: 230,036 miles. "
        "That's 9.2× the circumference of the Earth at the equator, "
        "or enough to reach the Moon and back — with 14,000 miles to spare."
    ),
    (
        2024,
        "Sable pipeline restarts",
        "bottom",
        "Nearly a decade after the Refugio spill, Sable Offshore Corp wins approval "
        "to restart the same coastal California pipeline infrastructure. The decision "
        "is fiercely contested by environmental groups and Santa Barbara County officials. "
        "Critics note the aging steel pipe hasn't been replaced — only inspected. "
        "The restart marks one of the longest shutdowns ever reversed for a US pipeline."
    ),
]

print(f"{len(events)} events loaded")

8 events loaded


In [4]:
# ── Color palette ─────────────────────────────────────────────────────────────
COLORS = {
    'crude':   '#D4380D',  # deep red-orange
    'rpp':     '#1677FF',  # blue  (gasoline/diesel/jet fuel)
    'hvl':     '#389E0D',  # green (propane/butane)
    'co2':     '#722ED1',  # purple
    'ethanol': '#FA8C16',  # amber
    'total':   '#262626',  # near-black
    'event_spill':  '#CF1322',  # red flag — spill/incident
    'event_open':   '#0958D9',  # blue flag — pipeline opened
    'event_trend':  '#389E0D',  # green flag — trend milestone
    'event_policy': '#722ED1',  # purple flag — policy/regulatory
}

# Assign event types for color coding
event_types = {
    'Kalamazoo River spill':       'event_spill',
    'Refugio spill (Santa Barbara)':'event_spill',
    'Sable pipeline restarts':     'event_policy',
    'Dakota Access opens':         'event_open',
    'Shale boom begins':           'event_trend',
    'Crude network peaks':         'event_trend',
    'Network peaks: 230,036 mi':   'event_trend',
    'Colonial Pipeline ransomware':'event_spill',
}

print('Color palette ready')

Color palette ready


In [5]:
fig = go.Figure()

# ── Commodity lines ────────────────────────────────────────────────────────────
commodity_config = [
    ('crude',   'Crude Oil',                    'solid',  2.5),
    ('rpp',     'Refined Products (gas/diesel)', 'solid',  2.5),
    ('hvl',     'Highly Volatile Liquids (propane/butane)', 'solid', 2),
    ('co2',     'CO₂ & Other',                  'dot',    1.5),
    ('ethanol', 'Fuel Ethanol',                 'dot',    1.5),
]

for key, label, dash, width in commodity_config:
    fig.add_trace(go.Scatter(
        x=miles['year'], y=miles[key],
        name=label,
        line=dict(color=COLORS[key], dash=dash, width=width),
        hovertemplate=f'<b>{label}</b><br>%{{x}}: %{{y:,.0f}} miles<extra></extra>',
        legendgroup=key,
    ))

# ── Total miles line ───────────────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=miles['year'], y=miles['total'],
    name='Total Network',
    line=dict(color=COLORS['total'], width=3.5, dash='solid'),
    hovertemplate='<b>Total Network</b><br>%{x}: %{y:,.0f} miles<extra></extra>',
    legendgroup='total',
))

# ── Event annotations ──────────────────────────────────────────────────────────
for year, label, side, blurb in events:
    # Get the total mileage at that year for y-positioning
    row = miles[miles.year == year]
    if row.empty:
        continue
    y_val = float(row['total'].values[0])
    color = COLORS.get(event_types.get(label, 'event_trend'), '#555')

    y_offset = 8000 if side == 'top' else -8000
    ay_offset = 40 if side == 'top' else -40

    # Vertical dashed line
    fig.add_vline(
        x=year,
        line_width=1,
        line_dash='dash',
        line_color=color,
        opacity=0.45,
    )

    # Annotation arrow + label
    fig.add_annotation(
        x=year,
        y=y_val + y_offset,
        text=f"<b>{label}</b>",
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=1.5,
        arrowcolor=color,
        ax=0,
        ay=ay_offset,
        font=dict(size=10, color=color),
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor=color,
        borderwidth=1,
        borderpad=3,
        # Store blurb in customdata for future tooltip expansion
        hovertext=blurb,
    )

# ── Layout ─────────────────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text='<b>The Invisible Network: U.S. Hazardous Liquid Pipeline Miles, 2004–2025</b><br>'
             '<sup>Source: PHMSA Annual Report Mileage Summary (Form 7000-1.1)</sup>',
        font=dict(size=16),
        x=0.5, xanchor='center'
    ),
    xaxis=dict(
        title='Year',
        tickmode='linear', dtick=2,
        gridcolor='#f0f0f0',
    ),
    yaxis=dict(
        title='Pipeline Miles',
        tickformat=',',
        gridcolor='#f0f0f0',
    ),
    legend=dict(
        orientation='h',
        yanchor='bottom', y=1.02,
        xanchor='left', x=0,
    ),
    hovermode='x unified',
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=1100,
    height=650,
    margin=dict(l=60, r=40, t=120, b=60),
)

fig.show()
print('Visual 1A rendered')

Visual 1A rendered


---
## 3. Visual 1C — Scale Callout Card
"What even is this number?" — convert 228,000 miles into things people understand

In [6]:
total_2025 = float(miles[miles.year == 2025]['total'].values[0])

# Relatable comparisons
EARTH_CIRCUMFERENCE_MI = 24_901
MOON_DISTANCE_MI       = 238_855
INTERSTATE_MILES       = 48_786   # US Interstate Highway System
NY_LA_MI               = 2_791    # New York to Los Angeles

comparisons = [
    (f"{total_2025 / EARTH_CIRCUMFERENCE_MI:.1f}×",  "times around Earth's equator"),
    (f"{total_2025 / MOON_DISTANCE_MI:.2f}×",        "times the distance to the Moon"),
    (f"{total_2025 / INTERSTATE_MILES:.1f}×",        "times the entire US Interstate Highway System"),
    (f"{total_2025 / NY_LA_MI:.0f}×",               "trips from New York City to Los Angeles"),
]

print(f"\n{'='*55}")
print(f"  {total_2025:,.0f} miles of hazardous liquid pipeline")
print(f"{'='*55}")
for val, label in comparisons:
    print(f"  {val:>8}  {label}")
print()


  227,457 miles of hazardous liquid pipeline
      9.1×  times around Earth's equator
     0.95×  times the distance to the Moon
      4.7×  times the entire US Interstate Highway System
       81×  trips from New York City to Los Angeles



In [7]:
# Visual version: proportional bar comparing pipeline to Interstate Highway System
fig_scale = go.Figure()

items = [
    ('US Hazardous Liquid Pipelines\n(2025)', total_2025,    '#D4380D'),
    ('US Interstate Highway System',          INTERSTATE_MILES, '#1677FF'),
    ('New York → Los Angeles (×81)',          NY_LA_MI * 81, '#389E0D'),
]

fig_scale.add_trace(go.Bar(
    x=[v for _, v, _ in items],
    y=[n for n, _, _ in items],
    orientation='h',
    marker_color=[c for _, _, c in items],
    text=[f'{v:,.0f} mi' for _, v, _ in items],
    textposition='outside',
    hovertemplate='%{y}: %{x:,.0f} miles<extra></extra>',
))

fig_scale.update_layout(
    title=dict(
        text='<b>How Big Is 227,000 Miles?</b>',
        font=dict(size=15), x=0.5, xanchor='center'
    ),
    xaxis=dict(title='Miles', tickformat=',', gridcolor='#f0f0f0'),
    yaxis=dict(autorange='reversed'),
    plot_bgcolor='white', paper_bgcolor='white',
    width=800, height=280,
    margin=dict(l=240, r=100, t=60, b=40),
    showlegend=False,
)
fig_scale.show()

---
## 4. Load & Aggregate Part I — Installation Decade Data (2017–2025)

In [8]:
decade_cols = [
    'PARTIUNKWN', 'PARTIPRE20',
    'PARTI192029', 'PARTI193039', 'PARTI194049',
    'PARTI195059', 'PARTI196069', 'PARTI197079',
    'PARTI198089', 'PARTI199099',
    'PARTI200009', 'PARTI201019', 'PARTI202029'
]

frames = []
for year in range(2017, 2026):
    f = rf'{BASE}\HL AR {year} Part I.csv'
    df = pd.read_csv(f)
    for c in decade_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
    totals = df[decade_cols].sum().to_dict()
    totals['year'] = year
    frames.append(totals)

decade_df = pd.DataFrame(frames).set_index('year')

# Human-readable decade labels
decade_labels = {
    'PARTIUNKWN':  'Unknown',
    'PARTIPRE20':  'Pre-1920',
    'PARTI192029': '1920s',
    'PARTI193039': '1930s',
    'PARTI194049': '1940s',
    'PARTI195059': '1950s',
    'PARTI196069': '1960s',
    'PARTI197079': '1970s',
    'PARTI198089': '1980s',
    'PARTI199099': '1990s',
    'PARTI200009': '2000s',
    'PARTI201019': '2010s',
    'PARTI202029': '2020s',
}

decade_df = decade_df.rename(columns=decade_labels)

# 2025 snapshot as a Series
snap_2025 = decade_df.loc[2025].drop('Unknown')
total_2025_decade = snap_2025.sum()

print("\n2025 Snapshot:")
for dec, mi in snap_2025.items():
    pct = mi / total_2025_decade * 100
    bar = '█' * int(pct / 1.5)
    print(f"  {dec:10s} {mi:9,.0f} mi  {pct:5.1f}%  {bar}")
print(f"\n  TOTAL    {total_2025_decade:9,.0f} mi")


2025 Snapshot:
  Pre-1920         121 mi    0.1%  
  1920s          1,844 mi    0.8%  
  1930s          5,388 mi    2.4%  █
  1940s         14,890 mi    6.7%  ████
  1950s         32,155 mi   14.4%  █████████
  1960s         32,482 mi   14.5%  █████████
  1970s         28,803 mi   12.9%  ████████
  1980s         16,927 mi    7.6%  █████
  1990s         18,369 mi    8.2%  █████
  2000s         15,950 mi    7.1%  ████
  2010s         45,564 mi   20.4%  █████████████
  2020s         11,275 mi    5.0%  ███

  TOTAL      223,768 mi


---
## 5. Visual 3A — The Pipeline Timeline Strip

A single horizontal bar where segment width = miles installed in that decade.
Historical events float above each segment as context markers.

In [9]:
# Color gradient: older = more orange/red; newer = blue/green
decade_colors = {
    'Pre-1920': '#7F1D1D',
    '1920s':    '#991B1B',
    '1930s':    '#B91C1C',
    '1940s':    '#DC2626',
    '1950s':    '#F97316',
    '1960s':    '#FB923C',
    '1970s':    '#FCD34D',
    '1980s':    '#86EFAC',
    '1990s':    '#4ADE80',
    '2000s':    '#22C55E',
    '2010s':    '#16A34A',
    '2020s':    '#166534',
}

# Historical context labels shown above each decade segment
decade_context = {
    'Pre-1920': 'Before WWI',
    '1920s':    'Roaring Twenties',
    '1930s':    'Great Depression',
    '1940s':    'WWII era',
    '1950s':    'Eisenhower / Post-war boom',
    '1960s':    'Apollo program era',
    '1970s':    'OPEC oil crisis',
    '1980s':    'Deregulation era',
    '1990s':    'Internet age begins',
    '2000s':    'Post-9/11 / Iraq War',
    '2010s':    'Shale boom',
    '2020s':    'COVID / Energy transition',
}

# Age in 2025
decade_age = {
    'Pre-1920': '100+ years old',
    '1920s':    '95–105 years old',
    '1930s':    '85–95 years old',
    '1940s':    '75–85 years old',
    '1950s':    '65–75 years old',
    '1960s':    '55–65 years old',
    '1970s':    '45–55 years old',
    '1980s':    '35–45 years old',
    '1990s':    '25–35 years old',
    '2000s':    '15–25 years old',
    '2010s':    '5–15 years old',
    '2020s':    'Under 5 years old',
}

# Key regulations / safety laws for context
decade_safety = {
    'Pre-1920': 'No federal pipeline safety law',
    '1920s':    'No federal pipeline safety law',
    '1930s':    'No federal pipeline safety law',
    '1940s':    'No federal pipeline safety law',
    '1950s':    'No federal pipeline safety law',
    '1960s':    'No federal pipeline safety law',  
    '1970s':    'Natural Gas Pipeline Safety Act (1968) only recently passed',
    '1980s':    'Hazardous Liquid Pipeline Safety Act passed (1979)',
    '1990s':    'Post-Exxon Valdez reforms; Pipeline Safety Act (1992)',
    '2000s':    'Pipeline Safety Improvement Act (2002)',
    '2010s':    'Post-Kalamazoo reforms; PIPES Act (2016)',
    '2020s':    'PIPES Act (2020); increased inspection requirements',
}

print("Context data loaded")

Context data loaded


In [10]:
ordered_decades = [
    'Pre-1920','1920s','1930s','1940s','1950s','1960s',
    '1970s','1980s','1990s','2000s','2010s','2020s'
]

fig_strip = go.Figure()

x_cursor = 0
bar_height = 1.0

for dec in ordered_decades:
    if dec not in snap_2025.index:
        continue
    mi = snap_2025[dec]
    pct = mi / total_2025_decade * 100
    color = decade_colors.get(dec, '#888')
    
    hover = (
        f"<b>{dec}</b><br>"
        f"{mi:,.0f} miles  ({pct:.1f}% of network)<br>"
        f"{decade_age.get(dec, '')}<br>"
        f"<i>{decade_context.get(dec, '')}</i><br>"
        f"<i>{decade_safety.get(dec, '')}</i>"
    )

    # Main bar segment
    fig_strip.add_trace(go.Bar(
        x=[mi],
        y=['Pipeline Age'],
        orientation='h',
        base=x_cursor,
        marker_color=color,
        marker_line=dict(color='white', width=1),
        name=dec,
        hovertemplate=hover + '<extra></extra>',
        text=dec if pct > 3 else '',
        textposition='inside',
        textfont=dict(color='white', size=10),
    ))

    x_cursor += mi

# Key threshold lines
pre_1980_miles = sum(snap_2025[d] for d in ordered_decades if d in snap_2025.index and 
                     int(d.replace('Pre-1920','1900').replace('s','0').split('-')[0]) < 1980)
pre_1970_miles = sum(snap_2025[d] for d in ordered_decades if d in snap_2025.index and 
                     int(d.replace('Pre-1920','1900').replace('s','0').split('-')[0]) < 1970)

for threshold_mi, threshold_label, threshold_color in [
    (pre_1970_miles, f'Pre-1970: {pre_1970_miles:,.0f} mi ({pre_1970_miles/total_2025_decade*100:.0f}%)', '#B91C1C'),
    (pre_1980_miles, f'Pre-1980: {pre_1980_miles:,.0f} mi ({pre_1980_miles/total_2025_decade*100:.0f}%)', '#F97316'),
]:
    fig_strip.add_vline(
        x=threshold_mi,
        line_width=2, line_dash='dash', line_color=threshold_color,
        annotation_text=f'<b>{threshold_label}</b>',
        annotation_position='top',
        annotation_font=dict(color=threshold_color, size=10),
    )

fig_strip.update_layout(
    title=dict(
        text='<b>When Was It Built? U.S. Hazardous Liquid Pipeline — Mileage by Installation Decade (2025)</b><br>'
             '<sup>Each segment width = miles of active pipeline installed in that decade. Hover for details.</sup>',
        font=dict(size=14), x=0.5, xanchor='center'
    ),
    xaxis=dict(title='Cumulative Miles', tickformat=',', gridcolor='#f0f0f0'),
    yaxis=dict(visible=False),
    barmode='stack',
    plot_bgcolor='white', paper_bgcolor='white',
    width=1100, height=280,
    margin=dict(l=40, r=40, t=100, b=50),
    legend=dict(
        orientation='h',
        yanchor='bottom', y=-0.55,
        xanchor='center', x=0.5,
        font=dict(size=9)
    ),
    showlegend=True,
)

fig_strip.show()

print(f"\nKey callouts:")
print(f"  Pre-1970 pipe (55+ years old): {pre_1970_miles:,.0f} miles = {pre_1970_miles/total_2025_decade*100:.0f}%")
print(f"  Pre-1980 pipe (45+ years old): {pre_1980_miles:,.0f} miles = {pre_1980_miles/total_2025_decade*100:.0f}%")
print(f"  EPA founded 1970 — {pre_1970_miles/total_2025_decade*100:.0f}% of the network predates it")


Key callouts:
  Pre-1970 pipe (55+ years old): 121 miles = 0%
  Pre-1980 pipe (45+ years old): 121 miles = 0%
  EPA founded 1970 — 0% of the network predates it


---
## 6. Visual 3C — Age Tier Donut (simplified for general audience)

Four buckets that tell the story simply: *before the EPA / before modern regulations / recent*

In [11]:
# Bucket definitions (installation decade → age tier in 2025)
tier_map = {
    'Pre-1920': 'Pre-1970 (55+ yrs)',
    '1920s':    'Pre-1970 (55+ yrs)',
    '1930s':    'Pre-1970 (55+ yrs)',
    '1940s':    'Pre-1970 (55+ yrs)',
    '1950s':    'Pre-1970 (55+ yrs)',
    '1960s':    'Pre-1970 (55+ yrs)',
    '1970s':    '1970–1989 (35–55 yrs)',
    '1980s':    '1970–1989 (35–55 yrs)',
    '1990s':    '1990–2009 (15–35 yrs)',
    '2000s':    '1990–2009 (15–35 yrs)',
    '2010s':    'Post-2010 (under 15 yrs)',
    '2020s':    'Post-2010 (under 15 yrs)',
}

tier_context = {
    'Pre-1970 (55+ yrs)':        'Before the EPA, Clean Water Act, or any federal hazardous liquid pipeline safety law',
    '1970–1989 (35–55 yrs)':     'Early regulatory era — basic federal oversight just beginning',
    '1990–2009 (15–35 yrs)':     'Post-Exxon Valdez reform era — modern standards taking shape',
    'Post-2010 (under 15 yrs)':  'Modern construction — shale boom pipeline surge',
}

tier_totals = {}
for dec, mi in snap_2025.items():
    tier = tier_map.get(dec)
    if tier:
        tier_totals[tier] = tier_totals.get(tier, 0) + mi

tier_order = [
    'Pre-1970 (55+ yrs)',
    '1970–1989 (35–55 yrs)',
    '1990–2009 (15–35 yrs)',
    'Post-2010 (under 15 yrs)',
]
tier_colors = ['#B91C1C', '#F97316', '#86EFAC', '#166534']

labels = tier_order
values = [tier_totals[t] for t in tier_order]
pcts   = [v/total_2025_decade*100 for v in values]

fig_donut = go.Figure(go.Pie(
    labels=labels,
    values=values,
    hole=0.55,
    marker_colors=tier_colors,
    customdata=[[tier_context[t], f'{v:,.0f} miles', f'{p:.1f}%'] for t, v, p in zip(tier_order, values, pcts)],
    hovertemplate=(
        '<b>%{label}</b><br>'
        '%{customdata[1]}  (%{customdata[2]})<br>'
        '<i>%{customdata[0]}</i><extra></extra>'
    ),
    textinfo='percent+label',
    textfont=dict(size=11),
))

fig_donut.update_layout(
    title=dict(
        text='<b>How Old Is the U.S. Hazardous Liquid Pipeline Network?</b><br>'
             f'<sup>{total_2025_decade:,.0f} total miles — 2025 data</sup>',
        font=dict(size=14), x=0.5, xanchor='center'
    ),
    annotations=[dict(
        text=f'<b>{pre_1970_miles/total_2025_decade*100:.0f}%</b><br>predates<br>the EPA',
        x=0.5, y=0.5,
        font=dict(size=14, color='#B91C1C'),
        showarrow=False
    )],
    plot_bgcolor='white', paper_bgcolor='white',
    width=650, height=520,
    margin=dict(l=40, r=40, t=100, b=40),
    legend=dict(orientation='h', yanchor='bottom', y=-0.15, xanchor='center', x=0.5)
)

fig_donut.show()

for t, v, p in zip(tier_order, values, pcts):
    print(f"  {t:35s} {v:9,.0f} mi  {p:.1f}%")

  Pre-1970 (55+ yrs)                     86,880 mi  38.8%
  1970–1989 (35–55 yrs)                  45,730 mi  20.4%
  1990–2009 (15–35 yrs)                  34,319 mi  15.3%
  Post-2010 (under 15 yrs)               56,839 mi  25.4%


---
## 7. Aging Trend: Has Anything Changed 2017→2025?

How much of the old pipe is being replaced vs. how much new pipe is being added?

In [12]:
# Track pre-1970 miles year-over-year
old_decades = ['Pre-1920','1920s','1930s','1940s','1950s','1960s']

yearly_old = []
for yr in range(2017, 2026):
    row = decade_df.loc[yr]
    old_mi = sum(row.get(d, 0) for d in old_decades)
    total_mi = sum(row.get(d, 0) for d in ordered_decades if d in row.index)
    yearly_old.append({'year': yr, 'pre_1970_mi': old_mi, 'total_mi': total_mi,
                        'pre_1970_pct': old_mi / total_mi * 100 if total_mi > 0 else 0})

old_df = pd.DataFrame(yearly_old)

fig_trend = make_subplots(specs=[[{'secondary_y': True}]])

fig_trend.add_trace(
    go.Bar(x=old_df.year, y=old_df.pre_1970_mi,
           name='Pre-1970 miles (absolute)',
           marker_color='#F97316', opacity=0.75,
           hovertemplate='%{x}: %{y:,.0f} miles<extra></extra>'),
    secondary_y=False
)

fig_trend.add_trace(
    go.Scatter(x=old_df.year, y=old_df.pre_1970_pct,
               name='Pre-1970 share of total (%)',
               line=dict(color='#B91C1C', width=2.5),
               mode='lines+markers',
               hovertemplate='%{x}: %{y:.1f}% of network<extra></extra>'),
    secondary_y=True
)

fig_trend.update_layout(
    title=dict(
        text='<b>Pre-1970 Pipeline Miles Still in Active Service, 2017–2025</b><br>'
             '<sup>Is the oldest infrastructure being replaced — or just growing as a share?</sup>',
        font=dict(size=13), x=0.5, xanchor='center'
    ),
    xaxis=dict(title='Year', tickmode='linear', dtick=1, gridcolor='#f0f0f0'),
    plot_bgcolor='white', paper_bgcolor='white',
    width=900, height=420,
    margin=dict(l=60, r=80, t=100, b=60),
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='left', x=0),
)
fig_trend.update_yaxes(title_text='Miles of Pre-1970 Pipe', secondary_y=False,
                        tickformat=',', gridcolor='#f0f0f0')
fig_trend.update_yaxes(title_text='% of Total Network', secondary_y=True,
                        ticksuffix='%', showgrid=False)

fig_trend.show()

delta = old_df.loc[old_df.year==2025,'pre_1970_mi'].values[0] - old_df.loc[old_df.year==2017,'pre_1970_mi'].values[0]
delta_pct = old_df.loc[old_df.year==2025,'pre_1970_pct'].values[0] - old_df.loc[old_df.year==2017,'pre_1970_pct'].values[0]
print(f"\nPre-1970 miles change 2017→2025: {delta:+,.0f} miles  ({delta_pct:+.1f} pct pts of total network)")


Pre-1970 miles change 2017→2025: -5,725 miles  (-5.3 pct pts of total network)
